# NSU BDA. 2024 Accidents
Соревнование для студенттов курса АБМД, ФИТ НГУ 2024

Студент: Митюшин Владимир 24221

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

Загрузим данные

In [2]:
X_train = pd.read_csv('../data/X_train.csv', index_col=0)
X_testFinal = pd.read_csv('../data/X_test.csv', index_col=0)
Y_train = pd.read_csv('../data/Y_train.csv', index_col=0)

Выведем информацию по датасетам

In [3]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 499056 entries, 241270 to 121958
Data columns (total 35 columns):
 #   Column                                   Non-Null Count   Dtype 
---  ------                                   --------------   ----- 
 0   accident_index                           499056 non-null  object
 1   vehicle_reference                        499056 non-null  int64 
 2   casualty_class                           499056 non-null  int64 
 3   sex_of_casualty                          499056 non-null  int64 
 4   age_of_casualty                          499056 non-null  int64 
 5   age_band_of_casualty                     499056 non-null  int64 
 6   pedestrian_location                      499056 non-null  int64 
 7   pedestrian_movement                      499056 non-null  int64 
 8   car_passenger                            499056 non-null  int64 
 9   bus_or_coach_passenger                   499056 non-null  int64 
 10  pedestrian_road_maintenance_worker       499

In [4]:
Y_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 499056 entries, 241270 to 121958
Data columns (total 1 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   casualty_severity  499056 non-null  int64
dtypes: int64(1)
memory usage: 7.6 MB


Посмотрим более подробно на целевой признак

In [5]:
print("min: ", Y_train['casualty_severity'].min())
print("max: ", Y_train['casualty_severity'].max())
print("unique: ", Y_train['casualty_severity'].unique())

min:  1
max:  3
unique:  [3 2 1]


### Исходя из возможных значений целевого признака понимаем, что имеем задачу классификации

## Очистка и подготовка данных

In [6]:
X_train.describe()

,vehicle_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,...,speed_limit,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,...,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000,499056.000000
mean,1.457009,1.472198,1.366891,36.823471,6.312548,0.758696,0.613450,0.231964,0.050407,0.025949,...,37.549017,3.777957,1.678210,2.994840,223.466781,0.321896,1.104309,2.056154,1.638289,1.371584
std,2.323518,0.725208,0.529874,19.526246,2.453729,2.144590,1.963437,0.627427,0.436082,0.232426,...,14.742382,11.998430,2.516433,2.758235,935.872042,1.637604,2.372924,1.734888,1.790234,0.930508
min,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,1.000000,1.000000,1.000000,22.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,30.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000,34.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,30.000000,1.000000,2.000000,3.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
75%,2.000000,2.000000,2.000000,50.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,50.000000,3.000000,4.000000,6.000000,0.000000,0.000000,0.000000,4.000000,1.000000,2.000000
max,999.000000,3.000000,9.000000,102.000000,11.000000,10.000000,9.000000,9.000000,9.000000,2.000000,...,70.000000,99.000000,9.000000,6.000000,9999.000000,9.000000,9.000000,7.000000,9.000000,9.000000


In [7]:
df = X_train.copy()
df = df[df.select_dtypes('number').columns]
df = (df - df.mean())/df.std()
# df.fillna(df.median(), inplace=True)
df[X_train.select_dtypes('O').columns] = X_train.select_dtypes('O')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 499056 entries, 241270 to 121958
Data columns (total 35 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   vehicle_reference                        499056 non-null  float64
 1   casualty_class                           499056 non-null  float64
 2   sex_of_casualty                          499056 non-null  float64
 3   age_of_casualty                          499056 non-null  float64
 4   age_band_of_casualty                     499056 non-null  float64
 5   pedestrian_location                      499056 non-null  float64
 6   pedestrian_movement                      499056 non-null  float64
 7   car_passenger                            499056 non-null  float64
 8   bus_or_coach_passenger                   499056 non-null  float64
 9   pedestrian_road_maintenance_worker       499056 non-null  float64
 10  casualty_type                   

In [8]:
# df.hist(figsize=(20, 15));

In [9]:
# df[df.select_dtypes('number').columns].corr().style.background_gradient(cmap='coolwarm')

In [10]:
# df = df.drop(columns='accident_index')
# df = df.drop(columns=['age_of_casualty', 'pedestrian_movement', 'second_road_number', 'junction_control', 'bus_or_coach_passenger', 'pedestrian_road_maintenance_worker', 'vehicle_left_hand_drive', 'vehicle_type'])

df = df.drop(columns=['accident_index', 'vehicle_reference'])
df = df.drop(columns='age_of_casualty')
# df = df.drop(columns='pedestrian_movement')
# df = df.drop(columns='pedestrian_crossing_human_control')
# df = df.drop(columns='pedestrian_crossing_physical_facilities')
# df = df.drop(columns='engine_capacity_cc')
# df = df.drop(columns='bus_or_coach_passenger')
# df = df.drop(columns='vehicle_left_hand_drive') 

In [11]:
# import holoviews as hv
# from holoviews import dim
# from holoviews import opts
# hv.extension('bokeh')


# def f(x):
#     return hv.BoxWhisker(df[x]).opts(height=120, responsive=True, toolbar='above', invert_axes=True, tools=['hover'])

# hv.DynamicMap(f, kdims=['x']).redim.values(x=df.select_dtypes('number').columns)

In [12]:
# df.drop(df[df.vehicle_reference > 20].index, inplace=True)
# df['sex_of_casualty'].replace(9, -1) # df['sex_of_casualty'].median()
# df['car_passenger'].replace(9, -1) # df['car_passenger'].median()
# df['casualty_type'].replace([3, 4, 5, 22, 23, 97, 103, 104, 105, 106], 2).replace([20, 21, 98, 113], 19)

# df['sex_of_casualty'] = df['sex_of_casualty'].replace(9, -1).replace(-1, df['sex_of_casualty'].median())
# df['car_passenger'] = df['car_passenger'].replace(9, -1)
# df['casualty_type'] = df['casualty_type'].replace([3, 4, 5, 22, 23, 97, 103, 104, 105, 106], 2).replace([20, 21, 98, 113], 19)

### Посмотрим на категориальные признаки

In [13]:
df.describe(include='O')

,generic_make_model,local_authority_highway
count,499056,499056
unique,1130,210
top,-1,E10000016
freq,132305,16811


In [14]:
df['generic_make_model'] = df['generic_make_model'].str.split(n=1).str[0].str.upper()
# display(df['generic_make_model'].value_counts(dropna=False))

# df = df.drop(columns='generic_make_model')

In [15]:
df['local_authority_highway'] = df['local_authority_highway'].str[:4]
# display(df['local_authority_highway'].value_counts(dropna=False))

In [16]:
one_hot = pd.get_dummies(df.select_dtypes('O'), prefix=df.select_dtypes('O').columns, dtype=bool)
df = pd.concat([one_hot, df.select_dtypes('number'), df.select_dtypes('bool')], axis=1)

# df.describe()

In [17]:
# df.info()

## Перейдём к обучению

In [18]:
x_train, x_test, y_train, y_test = train_test_split(df, Y_train, test_size=0.1, random_state=42) # df \ X_train
print(x_train.shape, y_train.shape)

(449150, 151) (449150, 1)


In [24]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler()

x_train, y_train = rus.fit_resample(x_train, y_train)

In [25]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [26]:
print(x_train.shape, X_testFinal.shape)

(16464, 151) (166352, 35)


In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass  import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

model = OneVsRestClassifier(SVC(class_weight='balanced', probability=False, break_ties=True, verbose=True, random_state=42), n_jobs=-1, verbose=1)
# model = SVC(class_weight='balanced', probability=False, break_ties=True, verbose=True, random_state=42)
# model = LinearSVC(class_weight='balanced', random_state=42, verbose=1)
# model = OneVsRestClassifier(LinearSVC(class_weight='balanced', random_state=42, verbose=1), n_jobs=-1, verbose=1)

In [32]:
model.fit(x_train, y_train.values.ravel())

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:   49.5s finished


OneVsRestClassifier(estimator=SVC(break_ties=True, class_weight='balanced',
                                  random_state=42, verbose=True),
                    n_jobs=-1, verbose=1)

In [33]:
y_predicted_5 = model.predict(x_test)

In [35]:
f1_5 = f1_score(y_test, y_predicted_5, average='macro')
print(f1_5)

0.24680718429009052


In [ ]:
X_testFinal = X_testFinal.reindex(df.columns, axis=1, fill_value=0)
X_testFinal.describe()

,generic_make_model_-1,generic_make_model_ABARTH,generic_make_model_AJS,generic_make_model_ALEXANDER,generic_make_model_ALFA,generic_make_model_APRILIA,generic_make_model_AUDI,generic_make_model_BENELLI,generic_make_model_BENTLEY,generic_make_model_BMW,...,speed_limit,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,...,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,37.547730,3.767667,1.687151,3.008446,223.430250,0.313859,1.095220,2.051848,1.634257,1.371369
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.710353,11.917428,2.512841,2.759564,935.856416,1.618394,2.362987,1.732798,1.787258,0.933177
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,30.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,30.000000,2.000000,2.000000,3.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,50.000000,3.000000,4.000000,6.000000,0.000000,0.000000,0.000000,4.000000,1.000000,2.000000
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,70.000000,99.000000,9.000000,6.000000,9174.000000,9.000000,9.000000,7.000000,9.000000,9.000000


In [91]:
y_predicted_Final = model.predict(X_testFinal)

C:\Users\Horsen\AppData\Roaming\Python\Python39\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


In [92]:
print(y_predicted_Final)

[3 3 3 ... 3 3 3]


In [ ]:
# my_file = open("../predicts/log_reg_1.csv", "w+")

# my_file.write("Id,casualty_severity\n")
# my_i = 0
# for i, row in X_testFinal.iterrows():
#     my_file.write(f"{i}, {y_predicted_Final[my_i]}\n")
#     my_i = my_i+1
# my_file.close()